# KVO_07 — Nadine SAM manuscript: frozen ecological-results reader

**Purpose:** create publication-ready tables and figures from *existing* Baego crown-inventory outputs. **Read-only input; no model training, SAM inference, parameter tuning or alterations to historical outputs.**

**Ground rules**
- Primary benchmark: the six exhaustively digitized visible-crown ROIs in `ROI_Crowns_Kvo.shp`, references `Crown_Kristof.shp`.
- Nadine's `Crown_Nadine.shp` is sparse: use only annotated-crown recall / conditional geometric diagnostics, **never whole-ROI false positives or F1**.
- Six ROIs are **not automatically seven independent stand ages**. Do not label them by age until explicitly validated against field plot identifiers.
- Use the frozen, spatial leave-one-ROI-out Block 64 outputs and Block 65 ecological synthesis. Report aggregate and ROI-wise statistics; **no choosing a winner on the same holdouts**.
- Distinguish summed crown projected area from union canopy cover. Crowns visible from above are not the full stem inventory.
- This notebook creates artifacts in a **new** folder `KVO_work/KVO_07_Nadine_manuscript/`.
- Run top to bottom in Google Colab (CPU runtime sufficient).

In [ ]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
if not ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")
assert ROOT.exists(), f"Drive project root missing: {ROOT}"
SOURCE = ROOT / "KVO_work" / "KVO_06_UNet_guided_SAM3"
D60 = SOURCE / "experiments_v60_ecological_validation_all6"
D64 = SOURCE / "experiments_v64_sixfold_musanga_gradient"
D65 = SOURCE / "experiments_v65_scientific_synthesis"
DEST = ROOT / "KVO_work" / "KVO_07_Nadine_manuscript"
DEST.mkdir(parents=True,exist_ok=True)
print("Source:", SOURCE)
print("Writing new derived tables/figures only to:", DEST)

## 1. Audit saved input files

Stop if the frozen Block 64 outputs are missing. Run the *historical* notebook's saved-output production separately; do not silently replace frozen results or rerun models in this manuscript notebook. Block 65 tables are optional for basic summaries, but required for largest-crown analysis.

In [ ]:
sources = {
 "60_stands": D60/"all6_ecological_stand_metrics.csv",
 "64_frozen": D64/"all6_frozen_metrics.csv",
 "64_size": D64/"reference_size_recall.csv",
 "64_nadine": D64/"nadine_sparse_geometry.csv",
 "64_cover": D64/"musanga_cover_by_roi.csv",
 "65_methods": D65/"ecological_method_comparison.csv",
 "65_dominant": D65/"dominant_20_crowns_per_ha.csv",
 "65_size": D65/"crown_size_class_recovery.csv",
 "65_uncertainty": D65/"paired_six_roi_uncertainty.csv",
}
audit=pd.DataFrame([{"file":k,"exists":v.is_file(),"path":str(v)} for k,v in sources.items()])
display(audit)
audit.to_csv(DEST/"input_audit.csv",index=False)
assert sources["64_frozen"].is_file(), "Frozen Block 64 metrics are missing. Do not substitute experimental outputs."
tables={k:pd.read_csv(v) for k,v in sources.items() if v.is_file()}
for k,t in tables.items():
    print(k, "rows",len(t), "columns:",list(t.columns))
assert {"roi","model","method","iou_threshold","f1"}.issubset(tables["64_frozen"]), "Unexpected frozen metrics schema"

## 2. Freeze the comparison before interpreting

By default the descriptive summaries compare **raw SAM** and **confidence-based overlap filtering**, which do not require choosing a method from results. Learned RF variants are reported separately, explicitly labeled as exploratory spatial out-of-fold predictions. Change `PRIMARY_METHOD` only if decided *a priori*, not after looking at plots.

In [ ]:
M=tables["64_frozen"].copy()
M=M[np.isclose(pd.to_numeric(M["iou_threshold"],errors="coerce"),0.5)].copy()
assert not M.empty,"No IoU=0.5 records found."
assert M.duplicated(["roi","model","method"]).sum()==0,"Duplicated ROI/model/method."
print("Models:",sorted(M.model.astype(str).unique()))
print("Methods:",sorted(M.method.astype(str).unique()))
print("ROIs:",sorted(M.roi.unique()))
PRIMARY_METHOD = "confidence_nms" if "confidence_nms" in set(M.method) else "raw"
BASE = M[M.method==PRIMARY_METHOD].copy()
print("Fixed descriptive method:", PRIMARY_METHOD)
assert BASE.roi.nunique()==6, "Expected exactly six distinct ROI IDs"
metrics=[x for x in ["f1","count_bias_pct","sum_cpa_bias_pct","canopy_cover_error_pp","canopy_spatial_iou","pred_n","reference_n"] if x in M.columns]
display(BASE[["roi","model","method"]+metrics].sort_values(["model","roi"]))
BASE.to_csv(DEST/"table_1_roiwise_primary_method.csv",index=False)

## 3. Stand-level ecological reliability

**Do not pool crowns as independent stand replicates.** Six ROI-level observations supply descriptive uncertainty only. Bias is predicted minus reference, expressed as percent for count and summed CPA, and percentage points for canopy cover.

In [ ]:
agg={}
for k in ["f1","canopy_spatial_iou","count_bias_pct","sum_cpa_bias_pct","canopy_cover_error_pp"]:
    if k in BASE:
        agg[k+"_mean"]=(k,"mean")
        agg[k+"_median"]=(k,"median")
        if k.endswith("bias_pct") or k.endswith("error_pp"):
            agg[k+"_mean_absolute"]=(k,lambda s:float(np.abs(s).mean()))
summary=BASE.groupby("model",as_index=False).agg(**agg)
display(summary.round(3))
summary.to_csv(DEST/"table_2_primary_model_ecological_summary.csv",index=False)
if {"count_bias_pct","sum_cpa_bias_pct","canopy_cover_error_pp"}.issubset(BASE):
    fig,ax=plt.subplots(1,3,figsize=(13,4))
    for aa,col,lab in zip(ax,["count_bias_pct","sum_cpa_bias_pct","canopy_cover_error_pp"],["Crown count bias (%)","Summed CPA bias (%)","Canopy cover error (pp)"]):
        for model,g in BASE.groupby("model"):
            aa.scatter(g.roi,g[col],label=model)
        aa.axhline(0,color="gray",lw=1)
        aa.set(xlabel="Validation ROI",ylabel=lab)
    ax[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(DEST/"figure_2_ecological_bias_by_roi.png",dpi=220,bbox_inches="tight")
    plt.show()

## 4. Crown-size recovery and largest crowns

Use Block 64 crown-size recovery and Block 65 top-20/ha comparisons if available. The reference and prediction top-20 must be selected **independently**, as done in Block 65. Do not confuse strong large-crown accuracy with reliable recovery of the whole crown population.

In [ ]:
if "64_size" in tables:
    z=tables["64_size"].copy()
    z=z[np.isclose(pd.to_numeric(z.iou_threshold,errors="coerce"),.5)] if "iou_threshold" in z else z
    if "method" in z:z=z[z.method==PRIMARY_METHOD]
    display(z.head(20))
    z.to_csv(DEST/"table_3_crown_size_recovery.csv",index=False)
if "65_dominant" in tables:
    top=tables["65_dominant"].copy()
    if "method" in top:top=top[top.method==PRIMARY_METHOD]
    display(top.head(20))
    top.to_csv(DEST/"table_4_largest_20_per_ha.csv",index=False)
else:
    print("Block 65 largest-crown table not found; mark analysis PENDING, do not invent results.")

## 5. Human reference uncertainty and field metadata

The original manuscript contains independent information that is not recreated by these CSVs: the seven-stand chronosequence, field measurements, five expert annotators, QField reference crowns, acquisition methods, and comparison to conventional ITD methods. **Retain these sections only after verifying geographic alignment and metric definitions.** The six-ROI workflow must not be presented as a seven-stand chronosequence without a validated crosswalk.

In [ ]:
if "64_nadine" in tables:
    nd=tables["64_nadine"].copy()
    if "iou_threshold" in nd:nd=nd[np.isclose(pd.to_numeric(nd.iou_threshold,errors="coerce"),.5)]
    if "method" in nd:nd=nd[nd.method==PRIMARY_METHOD]
    display(nd.head(20))
    nd.to_csv(DEST/"table_5_nadine_sparse_reference_diagnostics.csv",index=False)
if "64_cover" in tables:
    cover=tables["64_cover"].copy()
    display(cover)
    cover.to_csv(DEST/"table_6_musanga_cover_descriptive.csv",index=False)
    print("Musanga cover describes structure; it is NOT a verified stand-age variable.")

## 6. Manuscript evidence ledger

**Status definitions**: VERIFIED_CSV = reproducible directly from saved outputs; ORIGINAL_MS = original Nadine manuscript only (not independently reconstructed here); NEEDS_CROSSWALK = cannot attach to six ROI IDs without field metadata; PENDING = not supported by present data products. This deliberately keeps original material while preventing accidental mixing of distinct study designs.

In [ ]:
ledger=[
 ("Study region and 2023 UAV acquisition","ORIGINAL_MS","Keep from Nadine manuscript; check flight height, DSM processing, and geolocation claims"),
 ("Seven stands and age/DBH/species inventory","NEEDS_CROSSWALK","Retain original context; require ROI→stand and plot geometry crosswalk before age-based modeling"),
 ("Five expert annotations and ICC","ORIGINAL_MS","Retain as independent uncertainty study; audit count denominators and ICC model"),
 ("QField independent field crowns","ORIGINAL_MS","Keep comparison but do not reuse as unbiased test after hyperparameter tuning"),
 ("Historical SAM1 implementation","ORIGINAL_MS","Reconcile original manuscript RGB/DSM/SLIC description with Block 53 standalone SAM1"),
 ("Six ROI exhaustive visible-crown references","VERIFIED_CSV","Check ROI polygons and completeness in archived notebook; reference is not full stem population"),
 ("ROI-level SAM ecological biases","VERIFIED_CSV","Tables 1–2 frozen Block 64 at IoU=0.5"),
 ("Size-dependent crown recall","VERIFIED_CSV" if "64_size" in tables else "PENDING","Table 3 at IoU=0.5"),
 ("Largest 20 crowns per hectare","VERIFIED_CSV" if "65_dominant" in tables else "PENDING","Table 4 from Block 65"),
 ("Age dependence across the six ROI benchmarks","NEEDS_CROSSWALK","Do not infer age from geographic location or Musanga fraction"),
 ("Classical ITD benchmark","ORIGINAL_MS","Retain compactly only after ensuring evaluation units and IoU definitions match"),
 ("Independent cross-forest generalization","PENDING","Luki external transfer not established by Baego spatial holdouts"),
]
L=pd.DataFrame(ledger,columns=["manuscript_component","evidence_status","editorial_action"])
display(L)
L.to_csv(DEST/"manuscript_evidence_ledger.csv",index=False)

## 7. Automatic result text for editing (not a full manuscript)

This creates an evidence-based Markdown insert with a provenance audit and tables. Edit the scientific narrative separately, preserving original sections that are not yet re-analyzed.

In [ ]:
def mdtable(df):
    try:return df.to_markdown(index=False,floatfmt=".3f")
    except ImportError:return df.to_csv(index=False)
lines=[
 "# Nadine SAM — ecological validation result insert",
 "",
 "Source: historical GitHub notebook KVO_06 blocks 60, 64 and 65; frozen saved Drive CSVs.",
 "No inference, model fitting or selection performed by this notebook.",
 f"Primary descriptive method (fixed in this reader): {PRIMARY_METHOD}.",
 "Unit of replication: six Baego visible-crown ROIs; no verified mapping to seven stand ages.",
 "Matching criterion: one-to-one IoU >= 0.5, as declared by Block 64.",
 "",
 "## Ecological results by model",
 mdtable(summary.round(3)),
 "",
 "## ROI-level findings",
 mdtable(BASE[["roi","model","method"]+metrics].round(3)),
 "",
 "## Evidence and manuscript sections",
 mdtable(L),
 "",
 "## Required before final claims",
 "1. Confirm provenance and exact parameterization of each selected SAM prediction file.",
 "2. Reconcile 6 ROI IDs with the 7 original chronosequence plots; do not conflate them.",
 "3. Check independent held-out status, confidence filtering and any model-selection leakage.",
 "4. Retain Nadine's original field/annotator/ITD material only with its own validated sample definitions.",
 "5. Report confidence intervals and limitations at ROI, not per-crown pseudoreplication level.",
]
(DEST/"manuscript_results_insert.md").write_text("\n".join(lines),encoding="utf-8")
print("Saved:",DEST/"manuscript_results_insert.md")
print("All original notebooks, reference polygons, model checkpoints and existing results remain untouched.")